# PersonaForge Colab Training

Run this notebook on a Colab T4 runtime. Keep secrets in Colab Secrets, not in the notebook.

In [ ]:
!nvidia-smi

## Clone or Enter Repo

If you already uploaded/cloned the repo, change `REPO_DIR` to that path.

In [ ]:
import os
REPO_URL = "https://github.com/Drscq/PersonaForge.git"
REPO_DIR = "/content/PersonaForge"
if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
%cd $REPO_DIR

In [ ]:
!python -m pip install --upgrade pip
!python -m pip install -e ".[dev]"

## Generate Local Synthetic Data

In [ ]:
!personaforge demo --out runs/demo --rounds 5 --samples-per-round 64
!sed -n '1,120p' runs/demo/report.md

## Install Training Stack

In [ ]:
!python -m pip install -e ".[train]"

## QLoRA SFT + DPO Smoke Run

The default config uses `max_steps=30` to verify the training path before scaling.

In [ ]:
!python -m personaforge.training.train_qlora --config configs/colab_qwen_0_5b.json --stage both

## Sanity Check Outputs

In [ ]:
!find adapters -maxdepth 2 -type f | head -50
!python -m personaforge.training.evaluate_adapter --dpo-data runs/demo/dpo.jsonl --out outputs/eval_summary.json

## Optional: Generate Adapter-vs-Base JudgeCal Pairs

In [ ]:
!python -m personaforge.training.evaluate_winrate --model-name Qwen/Qwen2.5-0.5B-Instruct --adapter-dir adapters/qwen2_5_0_5b_dpo --eval-data runs/demo/dpo.jsonl --out-dir outputs/winrate --limit 8
!find outputs/winrate -maxdepth 1 -type f -print